# cbj · 공통 앵커 학습과 퍼짐·분산 평가
준비된 다중 해상도 입력과 고정 canonical-group 5-fold를 그대로 사용합니다. 전체 train 6,201명의 OOF 확률, fold 간 성능 분산, 환자별 확률 퍼짐·과신, 클래스 내/간 확률 분산을 측정합니다.

비교 대상은 같은 분할의 학습 클래스 빈도(prior), 전체 변이 유전자 수만 사용하는 LR, 유전자 변이 유무만 사용하는 LR입니다. 앵커와 별도의 팀 전략을 만드는 것이 아니라 승격 판단을 위한 대조군입니다. 모든 LR 설정은 준비 단계의 고정값입니다.

**이번 실행은 OOF/cold-site 학습과 평가까지만 수행합니다.** 자동 기준을 통과해도 실제 출력과 GPT 검토 후에만 전체 재학습·비공개 Kaggle Model 등록을 진행합니다. test 예측·제출은 하지 않습니다.

In [ ]:
from pathlib import Path
import os,sys,json,hashlib
assert Path('/kaggle/input').exists()
os.environ['MPLCONFIGDIR']='/tmp/cbj_evaluation_mpl'
os.environ['OMP_NUM_THREADS']='4'
os.environ['OPENBLAS_NUM_THREADS']='4'
Path('/kaggle/working/cbj_anchor_evaluate.py').write_text('"""Train the frozen anchor and assess promotion; no full refit or registry write."""\nimport argparse,gzip,hashlib,json,time,warnings\nfrom pathlib import Path\nimport numpy as np\nimport pandas as pd\nfrom scipy.sparse import load_npz\nfrom sklearn.linear_model import LogisticRegression\nfrom sklearn.exceptions import ConvergenceWarning\nfrom sklearn.metrics import f1_score,accuracy_score,balanced_accuracy_score,log_loss,classification_report\nimport joblib\n\n\ndef probability_diagnostics(y,p,n_bins=10):\n    p=np.asarray(p,dtype=float);y=np.asarray(y,dtype=int);k=p.shape[1]\n    assert np.isfinite(p).all() and (p>=0).all() and np.allclose(p.sum(axis=1),1,atol=1e-5)\n    pred=p.argmax(axis=1);conf=p.max(axis=1);correct=pred==y\n    sorted_p=np.sort(p,axis=1)\n    entropy=-(p*np.log(np.clip(p,1e-15,1))).sum(axis=1)\n    brier=np.mean((p*p).sum(axis=1)-2*p[np.arange(len(y)),y]+1)\n    bins=np.minimum((conf*n_bins).astype(int),n_bins-1);rows=[];ece=0.\n    for b in range(n_bins):\n        ix=bins==b;n=int(ix.sum())\n        if n:\n            c=float(conf[ix].mean());a=float(correct[ix].mean());ece+=n/len(y)*abs(c-a)\n            rows.append({\'bin\':b,\'lower\':b/n_bins,\'upper\':(b+1)/n_bins,\'n\':n,\'mean_confidence\':c,\'accuracy\':a})\n    h=conf>=.9\n    summary={\'brier_multiclass_0_to_2\':float(brier),\'toplabel_ece_10_equal_width_bins\':float(ece),\n             \'normalized_entropy_mean\':float(np.mean(entropy/np.log(k))),\n             \'confidence_mean\':float(conf.mean()),\'top1_margin_mean\':float((sorted_p[:,-1]-sorted_p[:,-2]).mean()),\n             \'high_confidence_09_n\':int(h.sum()),\'high_confidence_09_accuracy\':float(correct[h].mean()) if h.any() else None}\n    detail=pd.DataFrame({\'predicted_index\':pred,\'confidence\':conf,\'correct\':correct,\'normalized_entropy\':entropy/np.log(k),\n                         \'effective_class_count\':np.exp(entropy),\'top1_margin\':sorted_p[:,-1]-sorted_p[:,-2],\n                         \'probability_variance_across_classes\':p.var(axis=1),\'true_class_probability\':p[np.arange(len(y)),y]})\n    return summary,pd.DataFrame(rows),detail\n\n\ndef metrics(y,p,k):\n    pred=p.argmax(axis=1)\n    return {\'macro_f1\':float(f1_score(y,pred,labels=np.arange(k),average=\'macro\',zero_division=0)),\n            \'accuracy\':float(accuracy_score(y,pred)),\'balanced_accuracy\':float(balanced_accuracy_score(y,pred)),\n            \'log_loss\':float(log_loss(y,p,labels=np.arange(k)))}\n\n\ndef fit_checked(x,y,params,name,out):\n    print(\'FIT START\',name,x.shape,flush=True);t=time.time();model=LogisticRegression(**params)\n    with warnings.catch_warnings(record=True) as ws:\n        warnings.simplefilter(\'always\',ConvergenceWarning);model.fit(x,y)\n    info={\'name\':name,\'converged\':not any(issubclass(w.category,ConvergenceWarning) for w in ws),\n          \'iterations\':int(model.n_iter_.max()),\'seconds\':time.time()-t,\'finite_coefficients\':bool(np.isfinite(model.coef_).all())}\n    joblib.dump(model,out/(name+\'.joblib\'),compress=3)\n    print(\'FIT END\',json.dumps(info),flush=True)\n    return model,info\n\n\ndef paired_group_bootstrap(y,model_pred,reference_pred,groups,k,reps=1000,seed=4201):\n    group_index=pd.factorize(groups,sort=True)[0];ng=group_index.max()+1;rng=np.random.default_rng(seed);values=[]\n    for _ in range(reps):\n        group_weight=np.bincount(rng.integers(0,ng,ng),minlength=ng);w=group_weight[group_index]\n        scores=[]\n        for pred in [model_pred,reference_pred]:\n            cm=np.bincount(y*k+pred,weights=w,minlength=k*k).reshape(k,k)\n            den=cm.sum(axis=0)+cm.sum(axis=1)\n            scores.append(np.divide(2*np.diag(cm),den,out=np.zeros(k),where=den>0).mean())\n        values.append([scores[0],scores[0]-scores[1]])\n    a=np.array(values)\n    return {\'replicates\':reps,\'unit\':\'canonical profile group\',\'conditional_on_fitted_OOF_models\':True,\n            \'anchor_macro_f1_ci95\':np.quantile(a[:,0],[.025,.975]).tolist(),\n            \'delta_macro_f1_ci95\':np.quantile(a[:,1],[.025,.975]).tolist()}\n\n\ndef main():\n    ap=argparse.ArgumentParser();ap.add_argument(\'--train\',action=\'store_true\');ap.add_argument(\'--prepared\',type=Path,required=True);ap.add_argument(\'--policy\',type=Path,required=True);ap.add_argument(\'--output\',type=Path,required=True)\n    args=ap.parse_args()\n    if not args.train or not Path(\'/kaggle/working\').exists():raise SystemExit(\'Explicit --train and Kaggle runtime required\')\n    root=args.prepared;out=args.output;out.mkdir(exist_ok=True,parents=True)\n    policy=json.loads(args.policy.read_text());pre=json.loads((root/\'preflight.json\').read_text());cfg=json.loads((root/\'anchor_config.json\').read_text())\n    assert hashlib.sha256((root/\'preflight.json\').read_bytes()).hexdigest()==policy[\'expected_preflight_sha256\']\n    for name,h in pre[\'file_sha256\'].items():assert hashlib.sha256((root/name).read_bytes()).hexdigest()==h,name\n    (out/\'promotion_policy.json\').write_text(json.dumps(policy,indent=2))\n    folds=pd.read_csv(root/\'fold_assignments.csv\');classes=cfg[\'class_order\'];k=len(classes);y=folds.SUBCLASS.map({c:i for i,c in enumerate(classes)}).to_numpy();n=len(y)\n    assert folds.groupby(\'canonical_hash\').fold.nunique().eq(1).all()\n    allp={name:np.full((n,k),np.nan) for name in [\'anchor\',\'prior\',\'burden\',\'gene_only\']};fit_reports=[];fold_rows=[]\n    for fold in range(cfg[\'n_folds\']):\n        tr=np.flatnonzero(folds.fold.ne(fold));va=np.flatnonzero(folds.fold.eq(fold))\n        assert set(folds.canonical_hash.iloc[tr]).isdisjoint(folds.canonical_hash.iloc[va])\n        with gzip.open(root/f\'fold_{fold}_encoder.json.gz\',\'rt\') as f:enc=json.load(f)\n        assert set(enc[\'fit_ids\'])==set(folds.ID.iloc[tr])\n        xt=load_npz(root/f\'fold_{fold}_train.npz\');xv=load_npz(root/f\'fold_{fold}_valid.npz\')\n        prior=(np.bincount(y[tr],minlength=k)+1)/(len(tr)+k);allp[\'prior\'][va]=prior\n        cols={\'anchor\':None,\'burden\':[enc[\'names\'].index(\'global|burden_log\')],\n              \'gene_only\':[i for i,name in enumerate(enc[\'names\']) if name.startswith(\'gene|\') and name.endswith(\'|nonWT\')]}\n        for name,indices in cols.items():\n            a=xt if indices is None else xt[:,indices];b=xv if indices is None else xv[:,indices]\n            model,info=fit_checked(a,y[tr],cfg[\'model_params\'],f\'{name}_fold_{fold}\',out)\n            assert list(model.classes_)==list(range(k));p=model.predict_proba(b);allp[name][va]=p;info[\'fold\']=fold;info[\'role\']=name;fit_reports.append(info)\n            row={\'fold\':fold,\'role\':name,\'train_n\':len(tr),\'valid_n\':len(va),**metrics(y[va],p,k)};fold_rows.append(row);print(\'SCORES\',json.dumps(row),flush=True)\n        fold_rows.append({\'fold\':fold,\'role\':\'prior\',\'train_n\':len(tr),\'valid_n\':len(va),**metrics(y[va],allp[\'prior\'][va],k)})\n        pd.DataFrame(fold_rows).to_csv(out/\'fold_metrics.csv\',index=False)\n        (out/\'fit_status.json\').write_text(json.dumps(fit_reports,indent=2))\n    evaluations={};diag_table=None\n    novel=pd.read_csv(root/\'oof_novelty_by_patient.csv\').set_index(\'ID\').loc[folds.ID].reset_index()\n    for name,p in allp.items():\n        d,cal,detail=probability_diagnostics(y,p);evaluations[name]={**metrics(y,p,k),**d}\n        table=folds.copy()\n        for j,cl in enumerate(classes):table[\'p_\'+cl]=p[:,j]\n        if name==\'anchor\':\n            for col in novel:\n                if col not in table:table[col]=novel[col].values\n            diag_table=pd.concat([table,detail],axis=1)\n        table.to_csv(out/(name+\'_oof_probabilities.csv\'),index=False)\n        cal.to_csv(out/(name+\'_calibration.csv\'),index=False)\n    diag_table.to_csv(out/\'anchor_uncertainty_by_patient.csv\',index=False)\n    pd.DataFrame(classification_report(y,allp[\'anchor\'].argmax(axis=1),labels=np.arange(k),target_names=classes,output_dict=True,zero_division=0)).T.to_csv(out/\'per_class.csv\')\n    fm=pd.DataFrame(fold_rows);fs=fm.loc[fm.role==\'anchor\',\'macro_f1\'].to_numpy()\n    variation={\'fold_f1_mean\':float(fs.mean()),\'fold_f1_std_ddof1\':float(fs.std(ddof=1)),\n               \'fold_f1_variance_ddof1\':float(fs.var(ddof=1)),\'fold_f1_min\':float(fs.min()),\'fold_f1_max\':float(fs.max()),\'fold_f1_range\':float(np.ptp(fs)),\n               \'note\':\'Folds are not independent repetitions; this measures split variability, not all training randomness.\'}\n    boot={name:paired_group_bootstrap(y,allp[\'anchor\'].argmax(axis=1),allp[name].argmax(axis=1),folds.canonical_hash,k,policy[\'bootstrap_replicates\']) for name in [\'burden\',\'gene_only\']}\n    # Dispersion is in comparable 26-class OOF probability space, not raw gene coordinates.\n    p=allp[\'anchor\'];center=p.mean(axis=0);within=between=0.;dispersion=[]\n    for cl in range(k):\n        z=p[y==cl];mean=z.mean(axis=0);w=float(((z-mean)**2).sum()/len(z));b=float(((mean-center)**2).sum())\n        within+=len(z)*w/n;between+=len(z)*b/n\n        dispersion.append({\'SUBCLASS\':classes[cl],\'n\':len(z),\'within_probability_variance_trace\':w,\'centroid_distance_squared_to_global\':b,\n                           \'mean_entropy\':float(diag_table.loc[y==cl,\'normalized_entropy\'].mean()),\'mean_confidence\':float(diag_table.loc[y==cl,\'confidence\'].mean())})\n    pd.DataFrame(dispersion).to_csv(out/\'class_dispersion.csv\',index=False)\n    variation.update(within_class_probability_variance=within,between_class_probability_variance=between,\n                     between_fraction_of_total=between/(within+between) if within+between else None)\n    # Natural novelty cohorts with per-class outcomes; zero-token rows are separate.\n    masks={\'no_variant\':novel.exact_n.eq(0),\'known_exact_only\':novel.exact_n.gt(0)&novel.exact_unseen.eq(0),\n           \'any_unseen_exact\':novel.exact_unseen.gt(0),\'mostly_unseen_exact\':novel.unseen_exact_fraction.ge(.5),\'any_unseen_site\':novel.site_unseen.gt(0)}\n    subgroup=[]\n    for group,mask in masks.items():\n        for cl in range(k):\n            ix=np.flatnonzero(mask.to_numpy()&(y==cl))\n            if len(ix):subgroup.append({\'group\':group,\'SUBCLASS\':classes[cl],\'n\':len(ix),\'recall\':float(np.mean(p[ix].argmax(axis=1)==cl)),\n                                       \'mean_true_class_log_loss\':float(-np.log(np.clip(p[ix,cl],1e-15,1)).mean())})\n    pd.DataFrame(subgroup).to_csv(out/\'novelty_class_metrics.csv\',index=False)\n    # Cold-site model uses exactly the saved split and train-only encoder.\n    cold=pd.read_csv(root/\'cold_site_split.csv\');cy=cold.SUBCLASS.map({c:i for i,c in enumerate(classes)}).to_numpy()\n    tr=np.flatnonzero(cold.split.eq(\'train\'));va=np.flatnonzero(cold.split.eq(\'cold_validation\'));present=sorted(set(cy[va]))\n    with gzip.open(root/\'cold_encoder.json.gz\',\'rt\') as f:enc=json.load(f)\n    assert \'site|BRAF|600\' not in enc[\'seen\'][\'site\'];assert set(enc[\'fit_ids\'])==set(cold.ID.iloc[tr])\n    xt=load_npz(root/\'cold_train.npz\');xv=load_npz(root/\'cold_valid.npz\');coldp={};cold_metrics={}\n    prior=(np.bincount(cy[tr],minlength=k)+1)/(len(tr)+k);coldp[\'prior\']=np.tile(prior,(len(va),1))\n    for name,indices in [(\'anchor\',None),(\'burden\',[enc[\'names\'].index(\'global|burden_log\')])]:\n        model,info=fit_checked(xt if indices is None else xt[:,indices],cy[tr],cfg[\'model_params\'],\'cold_\'+name,out)\n        info[\'role\']=\'cold_\'+name;fit_reports.append(info);coldp[name]=model.predict_proba(xv if indices is None else xv[:,indices])\n    for name,cp in coldp.items():\n        d,cal,detail=probability_diagnostics(cy[va],cp)\n        cold_metrics[name]={**metrics(cy[va],cp,k),**d,\'macro_f1_present_classes\':float(f1_score(cy[va],cp.argmax(axis=1),labels=present,average=\'macro\',zero_division=0))}\n        table=cold.iloc[va].copy().reset_index(drop=True)\n        for j,cl in enumerate(classes):table[\'p_\'+cl]=cp[:,j]\n        table.to_csv(out/(\'cold_\'+name+\'_probabilities.csv\'),index=False)\n    cold_context={\'train_n\':len(tr),\'validation_n\':len(va),\'present_classes\':[classes[i] for i in present],\n                  \'note\':\'Restricted BRAF600 stress scenario, seven validation classes. Not directly comparable with all-class OOF.\'}\n    a=evaluations[\'anchor\'];weak=max(evaluations[x][\'macro_f1\'] for x in [\'prior\',\'burden\'])\n    all_converged=all(i[\'converged\'] and i[\'finite_coefficients\'] for i in fit_reports)\n    gates={\n        \'all_fits_converged_finite\':all_converged,\n        \'beats_weak_controls\':a[\'macro_f1\']>=weak+policy[\'min_delta_vs_weak_controls\'],\n        \'burden_delta_ci_positive\':boot[\'burden\'][\'delta_macro_f1_ci95\'][0]>0,\n        \'gene_only_point_noninferiority\':a[\'macro_f1\']>=evaluations[\'gene_only\'][\'macro_f1\']-policy[\'gene_only_point_tolerance\'],\n        \'gene_only_ci_noninferiority\':boot[\'gene_only\'][\'delta_macro_f1_ci95\'][0]>-policy[\'gene_only_ci_tolerance\'],\n        \'fold_std\':variation[\'fold_f1_std_ddof1\']<=policy[\'max_fold_std\'],\n        \'worst_fold\':variation[\'fold_f1_min\']>=a[\'macro_f1\']-policy[\'max_worst_fold_gap\'],\n        \'probabilistic_scores_better_than_prior\':a[\'log_loss\']<evaluations[\'prior\'][\'log_loss\'] and a[\'brier_multiclass_0_to_2\']<evaluations[\'prior\'][\'brier_multiclass_0_to_2\'],\n        \'ece\':a[\'toplabel_ece_10_equal_width_bins\']<=policy[\'max_ece\'],\n        \'high_confidence_accuracy\':a[\'high_confidence_09_n\']<policy[\'min_high_conf_n\'] or a[\'high_confidence_09_accuracy\']>=policy[\'min_high_conf_accuracy\'],\n        \'cold_not_collapsed\':cold_metrics[\'anchor\'][\'macro_f1_present_classes\']>=max(cold_metrics[x][\'macro_f1_present_classes\'] for x in [\'prior\',\'burden\'])-policy[\'cold_f1_tolerance\'] and cold_metrics[\'anchor\'][\'log_loss\']<=policy[\'cold_logloss_prior_multiplier\']*cold_metrics[\'prior\'][\'log_loss\'],\n    }\n    gates={k:bool(v) for k,v in gates.items()}\n    report={\'evaluations\':evaluations,\'variation\':variation,\'paired_group_bootstrap\':boot,\'cold_metrics\':cold_metrics,\'cold_context\':cold_context,\n            \'fit_reports\':fit_reports,\'promotion_gates\':gates,\'automatic_gates_passed\':all(gates.values()),\n            \'registry_promotion_status\':\'AWAITING_GPT_AND_ARTIFACT_REVIEW\' if all(gates.values()) else \'HOLD_FAILED_GATES\',\n            \'full_train_refit_done\':False,\'n_oof\':n,\'class_order\':classes,\'policy_sha256\':hashlib.sha256(args.policy.read_bytes()).hexdigest(),\n            \'preflight_sha256\':policy[\'expected_preflight_sha256\'],\'new_external_data_performance\':\'NOT VERIFIED\'}\n    (out/\'assessment.json\').write_text(json.dumps(report,indent=2))\n    (out/\'fit_status.json\').write_text(json.dumps(fit_reports,indent=2))\n    print(\'ASSESSMENT\',json.dumps({k:report[k] for k in [\'evaluations\',\'variation\',\'promotion_gates\',\'registry_promotion_status\']}),flush=True)\n\n\nif __name__==\'__main__\':main()\n')
Path('/kaggle/working/cbj_promotion_policy.json').write_text('{\n  "version": "cbj-anchor-promotion-policy-v1",\n  "purpose": "private research anchor, not clinical deployment",\n  "expected_preflight_sha256": "8d62dca092bd652acaa8c99421dfe5dcbd443e288c1f69eba72aae6285567b15",\n  "bootstrap_replicates": 1000,\n  "min_delta_vs_weak_controls": 0.02,\n  "gene_only_point_tolerance": 0.01,\n  "gene_only_ci_tolerance": 0.02,\n  "max_fold_std": 0.05,\n  "max_worst_fold_gap": 0.1,\n  "max_ece": 0.15,\n  "min_high_conf_n": 30,\n  "min_high_conf_accuracy": 0.75,\n  "cold_f1_tolerance": 0.02,\n  "cold_logloss_prior_multiplier": 1.1,\n  "freeze": "Written before any new anchor training or outcome inspection. Do not relax after seeing results.",\n  "review": "Automatic gates are necessary, not sufficient. Inspect artifacts and consult GPT before full refit/registration.",\n  "limitations": "Engineering thresholds, not statistical/clinical guarantees. Five folds are dependent; bootstrap is conditional on fitted OOF models. Existing XGB .3162 uses a different split and is not a direct promotion gate."\n}')

policy_path=Path('/kaggle/working/cbj_promotion_policy.json');policy=json.loads(policy_path.read_text())
matches=[p for p in Path('/kaggle/input').rglob('preflight.json') if hashlib.sha256(p.read_bytes()).hexdigest()==policy['expected_preflight_sha256']]
assert len(matches)==1,'Expected frozen preparation v2 outputs'
ROOT=matches[0].parent;OUT=Path('/kaggle/working/cbj_anchor_assessment');OUT.mkdir(exist_ok=True)
from IPython.display import display,Markdown,Image
display(policy)
import subprocess
subprocess.run([sys.executable,'/kaggle/working/cbj_anchor_evaluate.py','--train','--prepared',str(ROOT),'--policy',str(policy_path),'--output',str(OUT)],check=True)

## 1. fold 간 성능의 퍼짐
전체 OOF Macro-F1과 fold별 점수의 평균은 다릅니다. 여기서는 둘을 구분하고 표준편차는 표본 표준편차(ddof=1)로 표시합니다. 같은 5-fold 안의 점수는 독립적인 반복 실험이 아닙니다. 그룹 부트스트랩도 이미 학습한 OOF 예측을 조건으로 한 표본 변동입니다.

In [ ]:
import numpy as np,pandas as pd,matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib import font_manager
font=next(Path('/kaggle/input').rglob('NanumGothic-Regular.ttf'));font_manager.fontManager.addfont(str(font))
plt.rcParams.update({'font.family':'NanumGothic','font.size':11,'axes.unicode_minus':False,'figure.dpi':120,'savefig.dpi':160})
report=json.loads((OUT/'assessment.json').read_text());fold=pd.read_csv(OUT/'fold_metrics.csv');df=pd.read_csv(OUT/'anchor_oof_probabilities.csv')
classes=report['class_order'];p=df[['p_'+c for c in classes]].to_numpy();y=df.SUBCLASS.map({c:i for i,c in enumerate(classes)}).to_numpy()
diag=pd.read_csv(OUT/'anchor_uncertainty_by_patient.csv')
SCOPE=f'전체 train OOF n={len(df):,} · 동일 profile 학습/검증 분리 · test 제외'
def save(fig,name):
    fig.savefig(OUT/(name+'.png'),bbox_inches='tight',facecolor='white');plt.close(fig);display(Image(filename=str(OUT/(name+'.png'))))
fig,(ax,bx)=plt.subplots(1,2,figsize=(15,5.5))
colors={'anchor':'#256493','gene_only':'#bd8d39','burden':'#888888','prior':'#bbbbbb'}
for name in colors:
    z=fold[fold.role==name];ax.plot(z.fold,z.macro_f1,marker='o',label=name,color=colors[name])
ax.set_xticks(range(5));ax.set_ylim(0,1);ax.set_xlabel('fold');ax.set_ylabel('Macro-F1');ax.legend();ax.set_title('같은 분할의 fold별 성능')
names=list(colors);scores=[report['evaluations'][x]['macro_f1'] for x in names]
bx.bar(names,scores,color=[colors[x] for x in names]);bx.set_ylim(0,1);bx.set_ylabel('전체 OOF Macro-F1');bx.set_title('전체 OOF를 합쳐 계산한 성능')
for i,v in enumerate(scores):bx.text(i,v+.02,f'{v:.4f}',ha='center')
v=report['variation'];fig.suptitle('앵커 성능과 fold 간 분산',fontsize=18,y=1.02)
fig.text(.02,-.02,SCOPE+f' · 앵커 fold 평균={v["fold_f1_mean"]:.4f}, 표준편차={v["fold_f1_std_ddof1"]:.4f}, 범위={v["fold_f1_range"]:.4f}',fontsize=10)
fig.tight_layout();save(fig,'01_fold_variation');display(report['paired_group_bootstrap'])

## 2. 환자별 확률 퍼짐과 과신
엔트로피는 26개 클래스에 확률이 퍼지는 정도이며 0은 한 클래스 집중, 1은 균등입니다. 높은 엔트로피 자체를 실패로 판단하지 않습니다. margin은 가장 높은 두 확률의 차이입니다. 아래 세 지표는 정답·오답을 나누어 함께 봅니다.

In [ ]:
fig,axs=plt.subplots(1,3,figsize=(16,5))
for ax,col,title in zip(axs,['normalized_entropy','top1_margin','confidence'],['정규화 엔트로피','Top-1과 Top-2 확률 차이','가장 높은 예측 확률']):
    for correct,color,label in [(True,'#256493','정답'),(False,'#bd8d39','오답')]:
        z=diag.loc[diag.correct==correct,col];ax.hist(z,bins=np.linspace(0,1,21),histtype='step',linewidth=2,color=color,label=f'{label} n={len(z)}')
    ax.set_xlim(0,1);ax.set_xlabel(title);ax.set_ylabel('환자 수');ax.legend(fontsize=9)
fig.suptitle('예측 확률의 퍼짐: 정답과 오답',fontsize=18,y=1.02);fig.text(.02,-.02,SCOPE+' · 고정 폭 0.05 히스토그램',fontsize=10)
fig.tight_layout();save(fig,'02_prediction_spread')
cal=pd.read_csv(OUT/'anchor_calibration.csv')
fig,(ax,bx)=plt.subplots(1,2,figsize=(14,5.5))
ax.plot([0,1],[0,1],color='#999999',linestyle='--',label='예측 확률=실제 정답률')
ax.plot(cal.mean_confidence,cal.accuracy,'o-',color='#256493');ax.set_xlim(0,1);ax.set_ylim(0,1);ax.set_xlabel('구간 내 평균 최고 확률');ax.set_ylabel('구간 내 실제 정답률');ax.legend(fontsize=9)
for r in cal.itertuples():ax.annotate(f'n={r.n}',(r.mean_confidence,r.accuracy),xytext=(5,5),textcoords='offset points',fontsize=8)
bx.bar(cal.bin,cal.n,color='#256493');bx.set_xticks(range(10),[f'{i/10:.1f}~{(i+1)/10:.1f}' for i in range(10)],rotation=45);bx.set_xlabel('최고 확률 구간');bx.set_ylabel('환자 수')
a=report['evaluations']['anchor'];fig.suptitle('확률 보정 진단: 신뢰도 곡선과 분모',fontsize=17,y=1.02)
fig.text(.02,-.03,SCOPE+f' · ECE={a["toplabel_ece_10_equal_width_bins"]:.4f}; Brier(0~2)={a["brier_multiclass_0_to_2"]:.4f}; log loss={a["log_loss"]:.4f}\nECE는 top-label 10개 고정 구간. Brier/log loss는 보정과 분류 정보를 함께 반영.',fontsize=10)
fig.tight_layout();save(fig,'03_calibration')

## 3. 클래스 내부와 클래스 사이의 퍼짐
수치는 전체 26차원 OOF 확률 공간에서 계산합니다. PCA는 그림을 위한 2차원 투영이며 유전자 공간·생물학적 거리가 아닙니다. 아래 클래스쌍은 기존에 혼동과 상충 profile이 확인된 쌍을 미리 선택했습니다. 점이 분리돼 보여도 그것만으로 성능·인과를 확정하지 않습니다.

In [ ]:
from sklearn.decomposition import PCA
pc=PCA(n_components=2,random_state=42);coords=pc.fit_transform(p)
fig,axs=plt.subplots(1,2,figsize=(15,6))
for ax,pair in zip(axs,[('KIPAN','KIRC'),('GBMLGG','LGG')]):
    other=~df.SUBCLASS.isin(pair);ax.scatter(coords[other,0],coords[other,1],s=4,c='#dddddd',alpha=.2,label='기타 클래스')
    for cl,color in zip(pair,['#256493','#bd8d39']):
        mask=df.SUBCLASS==cl;ax.scatter(coords[mask,0],coords[mask,1],s=8,c=color,alpha=.4,label=f'{cl} n={mask.sum()}')
    ax.set_title(' / '.join(pair));ax.legend(fontsize=9);ax.set_xlabel(f'PC1 ({pc.explained_variance_ratio_[0]:.1%})');ax.set_ylabel(f'PC2 ({pc.explained_variance_ratio_[1]:.1%})')
fig.suptitle('OOF 확률 공간에서 클래스가 섞이는 정도',fontsize=17,y=1.02)
fig.text(.02,-.02,SCOPE+' · 투영 설명분산 표시; 전체 차원의 수치는 class_dispersion.csv',fontsize=10)
fig.tight_layout();save(fig,'04_class_probability_spread')
disp=pd.read_csv(OUT/'class_dispersion.csv');per=pd.read_csv(OUT/'per_class.csv',index_col=0).loc[classes]
fig,(ax,bx)=plt.subplots(1,2,figsize=(15,10),gridspec_kw={'width_ratios':[1,1.4]})
ax.barh(classes,per['f1-score'],color='#256493');ax.invert_yaxis();ax.set_xlim(0,1);ax.set_xlabel('OOF F1');ax.set_title('클래스별 성능')
bx.barh(disp.SUBCLASS,disp.within_probability_variance_trace,color='#256493');bx.invert_yaxis();bx.set_xlabel('클래스 내 확률 분산 합 (26차원)');bx.set_title('클래스 내부의 퍼짐')
fig.suptitle('클래스별 성능과 분산은 함께 해석',fontsize=17,y=1.01)
fig.text(.02,-.02,SCOPE+' · 낮은 분산이 항상 좋은 성능을 의미하지 않음. 클래스별 n은 per_class.csv에 보존.',fontsize=10)
fig.tight_layout();save(fig,'05_class_metrics_dispersion')

## 4. 신규 위치 스트레스와 승격 판정
BRAF 600을 가진 모든 profile을 학습에서 제외한 cold-site 결과입니다. 평가군은 339명·7개 클래스이므로 전체 OOF와 다른 모집단입니다. 같은 cold-site의 학습 prior·변이 수 대조군과 비교합니다.

자동 기준은 결과 확인 전에 고정한 연구용 기준입니다. 결과를 보고 완화하지 않습니다. 통과 후에도 GPT와 결과를 검토하고, 전체 train 재학습과 저장·추론 검증을 거쳐야 Kaggle Model로 등록합니다.

In [ ]:
display(pd.DataFrame(report['cold_metrics']).T)
display(pd.DataFrame([{'criterion':k,'passed':v} for k,v in report['promotion_gates'].items()]))
display(Markdown('**'+report['registry_promotion_status']+'**'))
display(report['variation'])
import shutil
shutil.copy2('/kaggle/working/cbj_anchor_evaluate.py',OUT/'cbj_anchor_evaluate.py')
print('Evaluation completed. No full-train refit, registry upload, or test prediction in this notebook.')